# vR.P.41 — Image Tampering Detection and Localization
## Unified Inception V1+V2+V3 custom encoder + Multi-Q RGB ELA (9ch)

---

| Field | Value |
|-------|-------|
| **Version** | vR.P.41 |
| **Track** | Pretrained Localization (Track 2) |
| **Architecture** | UNet with Unified Inception V1+V2+V3 Custom (from scratch) |
| **Change** | Unified Inception V1+V2+V3 custom encoder + Multi-Q RGB ELA (9ch) |
| **Parent** | vR.P.30.1 |
| **Dataset** | CASIA v2.0 — Splicing Detection + Localization (~12,614 images + GT masks) |
| **Platform** | Kaggle (T4/P100 GPU) |
| **Framework** | PyTorch + Segmentation Models PyTorch (SMP) |

---

### Pipeline Overview

```
Raw RGB Image
    |
    v
Multi-Quality RGB ELA Preprocessing (9ch)
    |-- JPEG recompress at Q=75 -> RGB ELA -> Channels 0–2
    |-- JPEG recompress at Q=85 -> RGB ELA -> Channels 3–5
    |-- JPEG recompress at Q=95 -> RGB ELA -> Channels 6–8
    |
    v
Unified Inception V1+V2+V3 Encoder (from scratch)
    |-- Stage 0: Input (9ch)
    |-- Stage 1: Stem Conv(32, stride=2) + BN -> 32ch
    |-- Stage 2: 2x InceptionV1 (classic, no BN) -> 240ch
    |-- Stage 3: 2x InceptionV2 (factorized 5x5, BN) -> 256ch
    |-- Stage 4: 2x InceptionV3 (asymmetric 1xN/Nx1, BN) -> 256ch
    |-- Stage 5: 1x InceptionV3 (n=5) -> 416ch
    |
    v
UNet Decoder (5 stages, skip connections from encoder)
    |
    v
Sigmoid → 384x384 Binary Mask
```

### Architecture Provenance
Ported from **Hitansh Inception DeepFake notebook** (TensorFlow/Keras) to PyTorch + SMP.
- Original: binary classification with GAP skip connections
- Ported: pixel-level segmentation with spatial skip connections via UNet
- Input: 9ch Multi-Q RGB ELA instead of 3ch RGB

### Input
- **Multi-Q RGB ELA → 9-Channel (Q=75/85/95 × R/G/B)**

---

## Change Log

| Version | Track | Change | Result |
|---------|-------|--------|--------|
| vR.P.3 | Pretrained | ELA as input Q=90 (replace RGB), frozen+BN | Pixel F1 = 0.6920 |
| vR.P.10 | Pretrained | CBAM attention + Focal+Dice | Pixel F1 = 0.7277 |
| vR.P.15 | Pretrained | Multi-quality ELA (Q=75/85/95) grayscale | Pixel F1 = 0.7329 |
| vR.P.19 | Pretrained | Multi-quality RGB ELA (9ch, Q=75/85/95) | Pixel F1 = 0.7965 |
| vR.P.30.1 | Pretrained | Multi-Q ELA + CBAM (50ep, BCE+Dice) | Pixel F1 = TBD |
| vR.P.40.1 | Pretrained | EfficientNet-B4 baseline (ELA Q=90, 3ch) | TBD |
| vR.P.40.2 | Pretrained | EfficientNet-B4 + Multi-Q RGB ELA (9ch) | TBD |
| vR.P.40.3 | Pretrained | InceptionV1 custom encoder + MQ-RGB-ELA 9ch | TBD |
| vR.P.40.4 | Pretrained | InceptionV2 custom encoder + MQ-RGB-ELA 9ch | TBD |
| vR.P.40.5 | Pretrained | InceptionV3 custom encoder + MQ-RGB-ELA 9ch | TBD |
| **vR.P.41** | Pretrained | **Unified Inception V1+V2+V3 custom encoder + MQ-RGB-ELA 9ch** | **TBD** |

In [ ]:
# ============================================================
# 1. SETUP
# ============================================================

# Install segmentation-models-pytorch
!pip install -q segmentation-models-pytorch
!pip install -q wandb

import os
import sys
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageChops, ImageEnhance
from io import BytesIO
from pathlib import Path
from tqdm.auto import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import transforms
import segmentation_models_pytorch as smp

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# ============================================================
# Configuration
# ============================================================
VERSION = 'vR.P.41'
CHANGE = 'Unified Inception V1+V2+V3 custom encoder + Multi-Q RGB ELA (9ch)'
SEED = 42
IMAGE_SIZE = 384
BATCH_SIZE = 16
ENCODER = 'unified-inception-v123'
ENCODER_WEIGHTS = 'imagenet'
IN_CHANNELS = 9
NUM_CLASSES = 1
LEARNING_RATE = 1e-4
ELA_QUALITIES = [75, 85, 95]
ATTENTION_TYPE = None
DECODER_CHANNELS = (256, 128, 64, 32, 16)
EPOCHS = 50
PATIENCE = 10
NUM_WORKERS = 0  # Kaggle safe
CHECKPOINT_PATH = f'{VERSION}_checkpoint.pth'
# --- Kaggle Persistence ---
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
RESULTS_DIR = '/kaggle/working/results'
LOGS_DIR = '/kaggle/working/logs'
for _d in [CHECKPOINT_DIR, RESULTS_DIR, LOGS_DIR]:
    os.makedirs(_d, exist_ok=True)
RESUME = True
LATEST_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pt')
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, 'best_model.pt')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Reproducibility
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything(SEED)

# --- Weights & Biases ---
USE_WANDB = True
WANDB_PROJECT = 'Tampered Image Detection & Localization'
DATASET_NAME = 'CASIA2'

import re as _re
_nb_dir = os.path.basename(os.getcwd()).lower()
_run_match = _re.search(r'run-(\d+)', _nb_dir)
RUN_ID = f'run{_run_match.group(1)}' if _run_match else 'run01'
EXPERIMENT_ID = VERSION.lower().replace('.', '').replace(' ', '')

_change_lower = CHANGE.lower()
_input_lower = globals().get('INPUT_TYPE', '').lower()
FEATURE_SET = 'rgb'
if 'multi' in _change_lower and 'rgb' in _change_lower:
    FEATURE_SET = 'multi-q-rgb-ela-9ch'
elif 'multi' in _change_lower or 'mq' in _change_lower:
    FEATURE_SET = 'multi-q-ela'
elif 'ela' in _change_lower:
    FEATURE_SET = 'ela'
elif 'dct' in _change_lower:
    FEATURE_SET = 'dct'


try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    if wandb_key:
        import wandb
        wandb.login(key=wandb_key)
        print("W&B logged in via Kaggle Secrets")
    else:
        USE_WANDB = False
except Exception:
    USE_WANDB = False
    print("W&B not available - logging locally only")

if USE_WANDB:
    import wandb
    wandb.init(
        project=WANDB_PROJECT,
        name=f'{VERSION} — {CHANGE}',
        config={
            'version': VERSION, 'change': CHANGE, 'encoder': ENCODER,
            'in_channels': IN_CHANNELS, 'image_size': IMAGE_SIZE,
            'batch_size': BATCH_SIZE, 'lr': LEARNING_RATE,
            'ela_qualities': ELA_QUALITIES, 'epochs': EPOCHS,
            'patience': PATIENCE, 'seed': SEED,
            'feature_set': FEATURE_SET, 'dataset': DATASET_NAME,
        },
        tags=[VERSION, ENCODER, FEATURE_SET, DATASET_NAME],
    )

print(f'{VERSION} — {CHANGE}')
print(f'Device: {DEVICE}')
print(f'Encoder: {ENCODER} | In channels: {IN_CHANNELS} | ELA Q: {ELA_QUALITIES}')

---

## 2. Dataset

### CASIA v2.0

The CASIA v2.0 dataset contains ~12,614 images:
- **Authentic (Au):** ~7,491 images â€” unmodified photographs
- **Tampered (Tp):** ~5,123 images â€” images with splicing or copy-move forgery

For **localization**, we need pixel-level ground truth masks indicating which regions are tampered. CASIA v2.0 includes ground truth masks for tampered images. If GT masks are not available in the Kaggle dataset, we generate pseudo-masks using ELA thresholding as a fallback.

For **authentic images**, the ground truth mask is all zeros (no tampering).

In [ ]:
# ============================================================
# 2.1 Dataset Path Discovery (FIXED in vR.P.1)
# ============================================================

def find_dataset():
    """Search /kaggle/input/ for Au/ and Tp/ directories.
    
    FIXED in vR.P.1: Collects ALL candidate dirs containing Au+Tp,
    then prefers the one with 'image' in the path (not 'mask').
    Also detects sibling MASK directory as ground truth.
    
    Returns: (dataset_root, au_dir, tp_dir, gt_mask_dir_or_None)
    """
    search_roots = ['/kaggle/input', '/content/drive/MyDrive']
    candidates = []  # list of (dirpath, au_path, tp_path)
    
    for base in search_roots:
        if not os.path.isdir(base):
            continue
        for dirpath, dirnames, _ in os.walk(base):
            if 'Au' in dirnames and 'Tp' in dirnames:
                candidates.append((
                    dirpath,
                    os.path.join(dirpath, 'Au'),
                    os.path.join(dirpath, 'Tp')
                ))
    
    if not candidates:
        return None, None, None, None
    
    # Separate IMAGE vs MASK candidates
    image_candidates = [c for c in candidates if 'mask' not in c[0].lower()]
    mask_candidates = [c for c in candidates if 'mask' in c[0].lower()]
    
    # Prefer IMAGE directory; fall back to first candidate if no IMAGE found
    if image_candidates:
        # Among image candidates, prefer the one with 'image' in path
        explicit_image = [c for c in image_candidates if 'image' in c[0].lower()]
        chosen = explicit_image[0] if explicit_image else image_candidates[0]
    else:
        chosen = candidates[0]
    
    # Detect GT mask directory
    gt_dir = None
    if mask_candidates:
        # Use the MASK candidate that has Au/Tp structure (for tampered masks)
        gt_dir = mask_candidates[0][0]  # root of MASK/{Au,Tp}
    
    return chosen[0], chosen[1], chosen[2], gt_dir

DATASET_ROOT, AU_DIR, TP_DIR, GT_DIR_ROOT = find_dataset()

if DATASET_ROOT is None:
    for base in ['/kaggle/input']:
        if os.path.isdir(base):
            print(f'Contents of {base}:')
            for dirpath, dirnames, filenames in os.walk(base):
                depth = dirpath.replace(base, '').count(os.sep)
                print(f'{"  " * depth}{os.path.basename(dirpath)}/')
                if depth >= 3:
                    break
    raise FileNotFoundError('Could not find Au/ and Tp/ directories.')

# Resolve GT mask directory
# GT_DIR_ROOT may be MASK/ which contains Au/ and Tp/ subdirs.
# For tampered images, GT masks are in MASK/Tp/
# We need to check if MASK/Tp/ contains actual mask images.
GT_DIR = None
if GT_DIR_ROOT is not None:
    # Check if MASK/Tp/ has image files (those are GT masks for tampered images)
    gt_tp_dir = os.path.join(GT_DIR_ROOT, 'Tp')
    if os.path.isdir(gt_tp_dir):
        gt_files = [f for f in os.listdir(gt_tp_dir)
                    if Path(f).suffix.lower() in {'.jpg', '.jpeg', '.png', '.tif', '.bmp'}]
        if gt_files:
            GT_DIR = GT_DIR_ROOT  # MASK directory with Au/Tp structure
            print(f'GT mask structure detected: {GT_DIR}')
            print(f'  MASK/Au: {len(os.listdir(os.path.join(GT_DIR, "Au")))} files')
            print(f'  MASK/Tp: {len(gt_files)} mask files')

# If GT_DIR_ROOT didn't work, search for other GT mask directories
if GT_DIR is None:
    # Search within dataset neighborhood
    search_base = os.path.dirname(DATASET_ROOT)
    for root, dirs, files in os.walk(search_base):
        for d in dirs:
            if any(pat in d.lower() for pat in ['groundtruth', 'gt', 'mask']):
                candidate = os.path.join(root, d)
                if any(Path(f).suffix.lower() in {'.jpg','.jpeg','.png','.tif','.bmp'}
                       for f in os.listdir(candidate) if os.path.isfile(os.path.join(candidate, f))):
                    GT_DIR = candidate
                    break
        if GT_DIR:
            break
    
    # Search separate Kaggle datasets
    if GT_DIR is None:
        input_dir = '/kaggle/input'
        if os.path.isdir(input_dir):
            for d in sorted(os.listdir(input_dir)):
                if any(pat in d.lower() for pat in ['groundtruth', 'gt', 'mask']):
                    for root, dirs, files in os.walk(os.path.join(input_dir, d)):
                        img_files = [f for f in files if Path(f).suffix.lower() in {'.jpg','.jpeg','.png','.tif','.bmp'}]
                        if img_files:
                            GT_DIR = root
                            break
                    if GT_DIR:
                        break

SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

print(f'\nDataset root:  {DATASET_ROOT}')
print(f'Authentic dir: {AU_DIR}  ({len(os.listdir(AU_DIR))} files)')
print(f'Tampered dir:  {TP_DIR}  ({len(os.listdir(TP_DIR))} files)')
if GT_DIR:
    print(f'GT mask dir:   {GT_DIR}')
else:
    print(f'GT mask dir:   NOT FOUND \u2014 will generate pseudo-masks from ELA')


In [ ]:
# ============================================================
# 2.2 Collect Image Paths and Ground Truth Masks (UPDATED)
# ============================================================

def collect_paths(directory):
    """Collect sorted image paths from a directory."""
    paths = []
    for f in sorted(os.listdir(directory)):
        if Path(f).suffix.lower() in SUPPORTED_EXTENSIONS:
            paths.append(os.path.join(directory, f))
    return paths

au_paths = collect_paths(AU_DIR)
tp_paths = collect_paths(TP_DIR)

# Build GT mask lookup
# The GT directory may have MASK/Au/ and MASK/Tp/ structure
# OR it may be a flat directory with mask files
gt_map = {}
if GT_DIR:
    gt_tp_dir = os.path.join(GT_DIR, 'Tp')
    gt_au_dir = os.path.join(GT_DIR, 'Au')
    
    # If GT has Au/Tp structure, collect from Tp subdirectory
    if os.path.isdir(gt_tp_dir):
        for f in sorted(os.listdir(gt_tp_dir)):
            if Path(f).suffix.lower() in SUPPORTED_EXTENSIONS:
                stem = Path(f).stem.lower()
                gt_map[stem] = os.path.join(gt_tp_dir, f)
        print(f'GT masks loaded from MASK/Tp/: {len(gt_map)}')
    else:
        # Flat directory
        for f in sorted(os.listdir(GT_DIR)):
            if Path(f).suffix.lower() in SUPPORTED_EXTENSIONS:
                stem = Path(f).stem.lower()
                gt_map[stem] = os.path.join(GT_DIR, f)
        print(f'GT masks loaded (flat): {len(gt_map)}')

# Match tampered images to GT masks
tp_with_gt = 0
tp_without_gt = 0
for tp in tp_paths:
    stem = Path(tp).stem.lower()
    # Try exact match and common variants
    variants = [stem, stem + '_gt', stem.replace('tp', 'gt'), stem.replace('Tp', 'Gt')]
    found = any(v in gt_map for v in variants)
    if found:
        tp_with_gt += 1
    else:
        tp_without_gt += 1

USE_GT_MASKS = GT_DIR is not None and tp_with_gt > len(tp_paths) * 0.5

print(f'\nAuthentic images:  {len(au_paths)}')
print(f'Tampered images:   {len(tp_paths)}')
print(f'Total:             {len(au_paths) + len(tp_paths)}')
print(f'Class ratio (Au:Tp): {len(au_paths)/len(tp_paths):.2f}:1')
if GT_DIR:
    print(f'\nTampered with GT mask:    {tp_with_gt}')
    print(f'Tampered without GT mask: {tp_without_gt}')
print(f'\nUsing GT masks: {USE_GT_MASKS}')
if not USE_GT_MASKS:
    print('  -> Will generate pseudo-masks from ELA thresholding')


In [ ]:
# ============================================================
# 2.3 ELA Pseudo-Mask Generation (Fallback)
# ============================================================

def compute_ela(image_path, quality=90):
    """Compute Error Level Analysis map."""
    original = Image.open(image_path).convert('RGB')
    buffer = BytesIO()
    original.save(buffer, 'JPEG', quality=quality)
    buffer.seek(0)
    resaved = Image.open(buffer)
    ela = ImageChops.difference(original, resaved)
    extrema = ela.getextrema()
    max_diff = max(val[1] for val in extrema)
    if max_diff == 0:
        max_diff = 1
    scale = 255.0 / max_diff
    ela = ImageEnhance.Brightness(ela).enhance(scale)
    return ela

def generate_pseudo_mask(image_path, quality=90, threshold=50):
    """Generate a binary pseudo-mask from ELA.
    Pixels with ELA brightness above threshold are marked as tampered.
    """
    ela = compute_ela(image_path, quality)
    ela_gray = np.array(ela.convert('L'))
    # Adaptive threshold: use mean + 2*std of the ELA map
    mean_val = ela_gray.mean()
    std_val = ela_gray.std()
    adaptive_thresh = max(threshold, mean_val + 2 * std_val)
    mask = (ela_gray > adaptive_thresh).astype(np.float32)
    return mask

def get_gt_mask(image_path, target_size):
    """Get ground truth mask for an image.
    - Authentic images: all-zero mask
    - Tampered images: GT mask if available, else ELA pseudo-mask
    """
    is_tampered = '/Tp/' in image_path or '\\Tp\\' in image_path

    if not is_tampered:
        # Authentic â€” all zeros (no tampering)
        return np.zeros((target_size, target_size), dtype=np.float32)

    if USE_GT_MASKS:
        # Try to find matching GT mask
        stem = Path(image_path).stem.lower()
        variants = [stem, stem + '_gt', stem.replace('tp', 'gt'), stem.replace('Tp', 'Gt')]
        for v in variants:
            if v in gt_map:
                mask = Image.open(gt_map[v]).convert('L')
                mask = mask.resize((target_size, target_size), Image.NEAREST)
                mask_arr = np.array(mask, dtype=np.float32)
                # Normalize to [0, 1]
                if mask_arr.max() > 1:
                    mask_arr = mask_arr / 255.0
                # Binarize
                mask_arr = (mask_arr > 0.5).astype(np.float32)
                return mask_arr

    # Fallback: ELA pseudo-mask
    try:
        mask = generate_pseudo_mask(image_path)
        mask_pil = Image.fromarray((mask * 255).astype(np.uint8))
        mask_pil = mask_pil.resize((target_size, target_size), Image.NEAREST)
        return np.array(mask_pil, dtype=np.float32) / 255.0
    except Exception:
        return np.zeros((target_size, target_size), dtype=np.float32)

# Quick test
test_au = au_paths[0]
test_tp = tp_paths[0]
mask_au = get_gt_mask(test_au, IMAGE_SIZE)
mask_tp = get_gt_mask(test_tp, IMAGE_SIZE)
print(f'Authentic mask â€” shape: {mask_au.shape}, sum: {mask_au.sum():.0f} (should be 0)')
print(f'Tampered mask  â€” shape: {mask_tp.shape}, sum: {mask_tp.sum():.0f} (should be > 0)')

---

## 3. Data Preparation

### Input Pipeline
- **Multi-quality ELA maps** computed on-the-fly from raw images
  - Channel 0: ELA at Q=75 (grayscale) --- strong compression, highlights major artifacts
  - Channel 1: ELA at Q=85 (grayscale) --- medium compression, balanced sensitivity
  - Channel 2: ELA at Q=95 (grayscale) --- light compression, reveals subtle edits
- Resize each channel to **384x384**
- **Per-channel normalization** (mean/std computed from training set)
- Binary segmentation masks (0=authentic, 1=tampered)
- **70/15/15** stratified split

### Why Multi-Quality ELA?

Different JPEG quality factors expose different artifact magnitudes:

| Quality | Compression | Residual Magnitude | Best For |
|---------|-------------|-------------------|----------|
| Q=75 | Aggressive | Large | Strong manipulations, copy-move |
| Q=85 | Medium | Medium | Balanced detection |
| Q=95 | Light | Small | Subtle splicing, fine boundaries |

Single-quality ELA (Q=90) provides one view. Multi-quality provides three complementary views, like RGB provides three color channels.

### Domain Adaptation
The frozen ResNet-34 encoder was trained on 3-channel RGB images. Multi-quality ELA also has 3 channels, so `conv1` accepts the input without modification. BatchNorm layers adapt to the new per-channel statistics.


In [ ]:
# ============================================================
# 3.1 PyTorch Dataset and Transforms (Multi-Q RGB ELA 9ch)
# ============================================================

def compute_ela_rgb(image_path, quality, size=384):
    """Compute ELA at given quality, return RGB image (H, W, 3)."""
    try:
        original = Image.open(image_path).convert('RGB')
        buffer = BytesIO()
        original.save(buffer, 'JPEG', quality=quality)
        buffer.seek(0)
        resaved = Image.open(buffer)
        ela = ImageChops.difference(original, resaved)
        extrema = ela.getextrema()
        max_diff = max(val[1] for val in extrema)
        if max_diff == 0:
            max_diff = 1
        scale = 255.0 / max_diff
        ela = ImageEnhance.Brightness(ela).enhance(scale)
        ela = ela.resize((size, size), Image.BILINEAR)
        return np.array(ela)  # (H, W, 3)
    except Exception:
        return np.zeros((size, size, 3), dtype=np.uint8)


def compute_multi_quality_rgb_ela(image_path, qualities=[75, 85, 95], size=384):
    """Stack RGB ELA at multiple quality levels -> (H, W, 9)."""
    channels = []
    for q in qualities:
        ela_rgb = compute_ela_rgb(image_path, q, size)
        channels.append(ela_rgb)
    return np.concatenate(channels, axis=-1)


def compute_mqela_statistics(image_paths, qualities=[75, 85, 95], n_samples=500, size=384):
    """Compute per-channel mean and std for 9-channel multi-Q RGB ELA."""
    rng = np.random.RandomState(42)
    indices = rng.choice(len(image_paths), min(n_samples, len(image_paths)), replace=False)

    channel_sum = np.zeros(9, dtype=np.float64)
    channel_sq_sum = np.zeros(9, dtype=np.float64)
    n_pixels = 0

    for idx in tqdm(indices, desc='MQ-RGB-ELA stats', leave=False):
        try:
            img = compute_multi_quality_rgb_ela(image_paths[idx], qualities, size)
            arr = img.astype(np.float64) / 255.0
            channel_sum += arr.reshape(-1, 9).sum(axis=0)
            channel_sq_sum += (arr.reshape(-1, 9) ** 2).sum(axis=0)
            n_pixels += arr.shape[0] * arr.shape[1]
        except Exception:
            continue

    mean = channel_sum / n_pixels
    std = np.sqrt(channel_sq_sum / n_pixels - mean ** 2)
    std = np.maximum(std, 1e-5)
    return mean.tolist(), std.tolist()


class CASIASegmentationDataset(Dataset):
    """CASIA v2.0 dataset for tampered region segmentation --- Multi-Q RGB ELA 9ch input."""

    def __init__(self, image_paths, labels, ela_mean, ela_std,
                 mask_size=384, qualities=None):
        self.image_paths = image_paths
        self.labels = labels
        self.mask_size = mask_size
        self.qualities = qualities or [75, 85, 95]
        self.normalize_mean = ela_mean
        self.normalize_std = ela_std

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        label = self.labels[idx]

        # Compute multi-quality RGB ELA (9 channels)
        try:
            mqela = compute_multi_quality_rgb_ela(path, self.qualities, self.mask_size)
        except Exception:
            mqela = np.zeros((self.mask_size, self.mask_size, 9), dtype=np.uint8)

        mqela = mqela.astype(np.float32) / 255.0
        mqela = torch.from_numpy(mqela).permute(2, 0, 1)  # (9, H, W)
        for c in range(9):
            mqela[c] = (mqela[c] - self.normalize_mean[c]) / self.normalize_std[c]

        mask = get_gt_mask(path, self.mask_size)
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        return mqela, mask, label

print(f'Dataset class ready (Multi-Q RGB ELA, Q={ELA_QUALITIES}, 9ch).')

In [ ]:
# ============================================================
# 3.2 Data Splitting (70/15/15 Stratified) + Multi-Q RGB ELA Stats
# ============================================================

all_paths = au_paths + tp_paths
all_labels = [0] * len(au_paths) + [1] * len(tp_paths)

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.30, stratify=all_labels, random_state=SEED)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, stratify=temp_labels, random_state=SEED)

# Compute multi-Q RGB ELA normalization statistics (9 channels)
ELA_MEAN, ELA_STD = compute_mqela_statistics(
    train_paths, qualities=ELA_QUALITIES, n_samples=500, size=IMAGE_SIZE)
print(f'Multi-Q RGB ELA normalization (9ch, from 500 training samples):')
for i, q in enumerate(ELA_QUALITIES):
    print(f'  Q={q} R/G/B mean: [{ELA_MEAN[i*3]:.4f}, {ELA_MEAN[i*3+1]:.4f}, {ELA_MEAN[i*3+2]:.4f}]  '
          f'std: [{ELA_STD[i*3]:.4f}, {ELA_STD[i*3+1]:.4f}, {ELA_STD[i*3+2]:.4f}]')

train_dataset = CASIASegmentationDataset(
    train_paths, train_labels, ela_mean=ELA_MEAN, ela_std=ELA_STD,
    mask_size=IMAGE_SIZE, qualities=ELA_QUALITIES)
val_dataset = CASIASegmentationDataset(
    val_paths, val_labels, ela_mean=ELA_MEAN, ela_std=ELA_STD,
    mask_size=IMAGE_SIZE, qualities=ELA_QUALITIES)
test_dataset = CASIASegmentationDataset(
    test_paths, test_labels, ela_mean=ELA_MEAN, ela_std=ELA_STD,
    mask_size=IMAGE_SIZE, qualities=ELA_QUALITIES)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=NUM_WORKERS > 0, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        persistent_workers=NUM_WORKERS > 0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         persistent_workers=NUM_WORKERS > 0)

print(f'\nTrain: {len(train_dataset):>6} images  (Au: {sum(1 for l in train_labels if l==0)}, Tp: {sum(1 for l in train_labels if l==1)})')
print(f'Val:   {len(val_dataset):>6} images  (Au: {sum(1 for l in val_labels if l==0)}, Tp: {sum(1 for l in val_labels if l==1)})')
print(f'Test:  {len(test_dataset):>6} images  (Au: {sum(1 for l in test_labels if l==0)}, Tp: {sum(1 for l in test_labels if l==1)})')
print(f'\nTrain batches: {len(train_loader)}  (drop_last=True)')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

In [ ]:
# ============================================================
# 3.3 Sample Visualization (Multi-Q RGB ELA Input)
# ============================================================

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
sample_indices = []
au_shown, tp_shown = 0, 0
for i in range(len(train_dataset)):
    if train_labels[i] == 0 and au_shown < 2:
        sample_indices.append(i); au_shown += 1
    elif train_labels[i] == 1 and tp_shown < 2:
        sample_indices.append(i); tp_shown += 1
    if au_shown >= 2 and tp_shown >= 2:
        break

for row, idx in enumerate(sample_indices):
    mqela_tensor, mask, label = train_dataset[idx]

    # Show Q=75 RGB (channels 0-2)
    q75 = mqela_tensor[:3].clone()
    for c in range(3):
        q75[c] = q75[c] * ELA_STD[c] + ELA_MEAN[c]
    axes[row, 0].imshow(q75.permute(1, 2, 0).clamp(0, 1).numpy())
    axes[row, 0].set_title(f'ELA Q=75 ({"Au" if label==0 else "Tp"})', fontsize=9)
    axes[row, 0].axis('off')

    # Show Q=85 RGB (channels 3-5)
    q85 = mqela_tensor[3:6].clone()
    for c in range(3):
        q85[c] = q85[c] * ELA_STD[c+3] + ELA_MEAN[c+3]
    axes[row, 1].imshow(q85.permute(1, 2, 0).clamp(0, 1).numpy())
    axes[row, 1].set_title('ELA Q=85', fontsize=9)
    axes[row, 1].axis('off')

    # Show Q=95 RGB (channels 6-8)
    q95 = mqela_tensor[6:9].clone()
    for c in range(3):
        q95[c] = q95[c] * ELA_STD[c+6] + ELA_MEAN[c+6]
    axes[row, 2].imshow(q95.permute(1, 2, 0).clamp(0, 1).numpy())
    axes[row, 2].set_title('ELA Q=95', fontsize=9)
    axes[row, 2].axis('off')

    try:
        orig = Image.open(train_paths[idx]).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
        axes[row, 3].imshow(np.array(orig))
    except Exception:
        axes[row, 3].text(0.5, 0.5, 'Load failed', ha='center', va='center')
    axes[row, 3].set_title('Original RGB', fontsize=9)
    axes[row, 3].axis('off')

    mask_display = mask.squeeze(0).numpy()
    axes[row, 4].imshow(mask_display, cmap='hot', vmin=0, vmax=1)
    axes[row, 4].set_title(f'GT Mask', fontsize=9)
    axes[row, 4].axis('off')

plt.suptitle(f'{VERSION} — Multi-Q RGB ELA Samples (9ch)', fontsize=14)
plt.tight_layout()
plt.show()

---

## 4. Model Architecture

### UNet with InceptionV1 Custom (from scratch)

Encoder: **inception-v1-custom** (trained from scratch)

In [ ]:
# ============================================================
# 4.1 Custom Unified Inception V1+V2+V3 Encoder + SMP Registration
# ============================================================

# --- InceptionV1 Module (Classic GoogLeNet, no BN) ---
class InceptionV1Module(nn.Module):
    """Classic Inception V1: 4 parallel branches (no BN). Ported from Hitansh notebook."""
    def __init__(self, in_ch, f1, f3r, f3, f5r, f5, fpool):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_ch, f1, 1), nn.ReLU(inplace=True))
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_ch, f3r, 1), nn.ReLU(inplace=True),
            nn.Conv2d(f3r, f3, 3, padding=1), nn.ReLU(inplace=True))
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_ch, f5r, 1), nn.ReLU(inplace=True),
            nn.Conv2d(f5r, f5, 5, padding=2), nn.ReLU(inplace=True))
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_ch, fpool, 1), nn.ReLU(inplace=True))
        self.out_channels = f1 + f3 + f5 + fpool

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x),
                         self.branch3(x), self.branch4(x)], dim=1)


# --- InceptionV2 Module (Factorized 5x5 -> two 3x3, BN, AvgPool) ---
class InceptionV2Module(nn.Module):
    """Inception V2: factorized 5x5 into two 3x3, BatchNorm, AvgPool. Ported from Hitansh notebook."""
    def __init__(self, in_ch, f1, f3r, f3, fdr, fd, fpool):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_ch, f1, 1, bias=False), nn.BatchNorm2d(f1), nn.ReLU(inplace=True))
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_ch, f3r, 1, bias=False), nn.BatchNorm2d(f3r), nn.ReLU(inplace=True),
            nn.Conv2d(f3r, f3, 3, padding=1, bias=False), nn.BatchNorm2d(f3), nn.ReLU(inplace=True))
        # Factorized 5x5 -> two 3x3
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_ch, fdr, 1, bias=False), nn.BatchNorm2d(fdr), nn.ReLU(inplace=True),
            nn.Conv2d(fdr, fd, 3, padding=1, bias=False), nn.BatchNorm2d(fd), nn.ReLU(inplace=True),
            nn.Conv2d(fd, fd, 3, padding=1, bias=False), nn.BatchNorm2d(fd), nn.ReLU(inplace=True))
        self.branch4 = nn.Sequential(
            nn.AvgPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_ch, fpool, 1, bias=False), nn.BatchNorm2d(fpool), nn.ReLU(inplace=True))
        self.out_channels = f1 + f3 + fd + fpool

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x),
                         self.branch3(x), self.branch4(x)], dim=1)


# --- InceptionV3 Module (Asymmetric 1xN + Nx1 factorization, BN) ---
class InceptionV3Module(nn.Module):
    """Inception V3: asymmetric 1xN + Nx1 factorization, BatchNorm.
    Ported from Hitansh Inception DeepFake notebook.
    Branches: b1=1x1, b2=1x1->1xN->Nx1, b3=1x1->1xN->Nx1->1xN->Nx1, b4=AvgPool->1x1."""
    def __init__(self, in_ch, f1, f3r, f3, fdr, fd, fpool, n=7):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_ch, f1, 1, bias=False), nn.BatchNorm2d(f1), nn.ReLU(inplace=True))
        # Single asymmetric: 1x1 -> 1xN -> Nx1
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_ch, f3r, 1, bias=False), nn.BatchNorm2d(f3r), nn.ReLU(inplace=True),
            nn.Conv2d(f3r, f3, (1, n), padding=(0, n // 2), bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(f3, f3, (n, 1), padding=(n // 2, 0), bias=False),
            nn.BatchNorm2d(f3), nn.ReLU(inplace=True))
        # Double asymmetric: 1x1 -> 1xN -> Nx1 -> 1xN -> Nx1
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_ch, fdr, 1, bias=False), nn.BatchNorm2d(fdr), nn.ReLU(inplace=True),
            nn.Conv2d(fdr, fd, (1, n), padding=(0, n // 2), bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(fd, fd, (n, 1), padding=(n // 2, 0), bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(fd, fd, (1, n), padding=(0, n // 2), bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(fd, fd, (n, 1), padding=(n // 2, 0), bias=False),
            nn.BatchNorm2d(fd), nn.ReLU(inplace=True))
        self.branch4 = nn.Sequential(
            nn.AvgPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_ch, fpool, 1, bias=False), nn.BatchNorm2d(fpool), nn.ReLU(inplace=True))
        self.out_channels = f1 + f3 + fd + fpool

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x),
                         self.branch3(x), self.branch4(x)], dim=1)


# --- Unified Inception V1+V2+V3 Encoder ---
from segmentation_models_pytorch.encoders._base import EncoderMixin

class UnifiedInceptionEncoder(nn.Module, EncoderMixin):
    """5-stage encoder using progressive Inception V1 -> V2 -> V3 modules.
    Ported from Hitansh 'Unified Inception V1+V2+V3 Slim' architecture.
    Original: classification with GAP skip connections.
    Ported: segmentation encoder with spatial feature maps for UNet decoder.

    Stage layout:
      Stage 0: Input (9ch, 384x384)
      Stage 1: Stem Conv(32, stride=2) + BN -> 32ch, 192x192
      Stage 2: 2x InceptionV1 + MaxPool -> 240ch, 96x96
      Stage 3: 2x InceptionV2 + MaxPool -> 256ch, 48x48
      Stage 4: 2x InceptionV3(n=3) + MaxPool -> 256ch, 24x24
      Stage 5: 1x InceptionV3(n=5) + MaxPool -> 416ch, 12x12"""

    def __init__(self, in_channels=9, depth=5, **kwargs):
        super().__init__()
        self._depth = depth
        self._in_channels = in_channels
        self._out_channels = [in_channels, 32, 240, 256, 256, 416]

        # Stage 1: Stem
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True))

        # Stage 2: Inception V1 (classic, no BN) -- 2 modules + MaxPool
        self.v1_m1 = InceptionV1Module(32, 32, 48, 64, 8, 16, 16)    # out=128
        self.v1_m2 = InceptionV1Module(128, 64, 64, 96, 16, 48, 32)  # out=240
        self.pool2 = nn.MaxPool2d(2, 2)

        # Stage 3: Inception V2 (factorized, BN) -- 2 modules + MaxPool
        self.v2_m1 = InceptionV2Module(240, 96, 48, 104, 8, 24, 32)   # out=256
        self.v2_m2 = InceptionV2Module(256, 80, 56, 112, 12, 32, 32)  # out=256
        self.pool3 = nn.MaxPool2d(2, 2)

        # Stage 4: Inception V3 (asymmetric, BN) -- 2 modules(n=3) + MaxPool
        self.v3_m1 = InceptionV3Module(256, 96, 48, 104, 8, 24, 32, n=3)   # out=256
        self.v3_m2 = InceptionV3Module(256, 80, 56, 112, 12, 32, 32, n=3)  # out=256
        self.pool4 = nn.MaxPool2d(2, 2)

        # Stage 5: Inception V3 (wider, n=5) -- 1 module + MaxPool
        self.v3_m3 = InceptionV3Module(256, 128, 80, 160, 16, 64, 64, n=5)  # out=416
        self.pool5 = nn.MaxPool2d(2, 2)

    @property
    def out_channels(self):
        return self._out_channels[:self._depth + 1]

    def forward(self, x):
        features = [x]                          # Stage 0: input (9, 384, 384)

        x = self.stem(x)                        # Stage 1: (32, 192, 192)
        features.append(x)

        x = self.v1_m1(x)                       # (128, 192, 192)
        x = self.v1_m2(x)                       # (240, 192, 192)
        x = self.pool2(x)                       # (240, 96, 96)
        features.append(x)                      # Stage 2

        x = self.v2_m1(x)                       # (256, 96, 96)
        x = self.v2_m2(x)                       # (256, 96, 96)
        x = self.pool3(x)                       # (256, 48, 48)
        features.append(x)                      # Stage 3

        x = self.v3_m1(x)                       # (256, 48, 48)
        x = self.v3_m2(x)                       # (256, 48, 48)
        x = self.pool4(x)                       # (256, 24, 24)
        features.append(x)                      # Stage 4

        x = self.v3_m3(x)                       # (416, 24, 24)
        x = self.pool5(x)                       # (416, 12, 12)
        features.append(x)                      # Stage 5

        return features[:self._depth + 1]

    def set_in_channels(self, in_channels, pretrained=False):
        """Override SMP default to avoid patch_first_conv on custom architecture."""
        if in_channels == self._in_channels:
            return
        self._in_channels = in_channels
        self._out_channels = [in_channels] + list(self._out_channels[1:])
        self.stem[0] = nn.Conv2d(in_channels, 32, 3, stride=2, padding=1, bias=False)

    def load_state_dict(self, state_dict, **kwargs):
        super().load_state_dict(state_dict, strict=False)


# Register with SMP
smp.encoders.encoders['unified-inception-v123'] = {
    'encoder': UnifiedInceptionEncoder,
    'pretrained_settings': {},
    'params': {'in_channels': IN_CHANNELS, 'depth': 5}
}

model = smp.Unet(
    encoder_name='unified-inception-v123',
    encoder_weights=None,
    in_channels=IN_CHANNELS,
    classes=NUM_CLASSES,
    decoder_channels=DECODER_CHANNELS,
    activation=None
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model: UNet + Unified Inception V1+V2+V3 Encoder (from scratch)')
print(f'  Encoder out_channels: {UnifiedInceptionEncoder(in_channels=IN_CHANNELS)._out_channels}')
print(f'  V1 stages: 2 modules (no BN, classic GoogLeNet)')
print(f'  V2 stages: 2 modules (factorized 5x5, BN)')
print(f'  V3 stages: 3 modules (asymmetric 1xN/Nx1, BN)')
print(f'Total parameters:     {total_params:>12,}')
print(f'Trainable parameters: {trainable_params:>12,} (100% \u2014 no pretrained weights)')

---

## 5. Training

### Configuration

| Parameter | Value |
|-----------|-------|
| Loss | BCE + Dice (combined) |
| Optimizer | Adam (single LR) |
| LR | **1e-3** (decoder + encoder BN) |
| Scheduler | ReduceLROnPlateau (factor=0.5, patience=3) |
| Early stopping | patience=7, monitor=val_loss |
| Epochs | 25 max |
| Batch size | 16 |

### Why Single LR (Not Differential)?

In vR.P.2, differential LR was needed because conv weights in layer3+4 were unfrozen.
In vR.P.3, all conv weights are frozen. Only BatchNorm params and decoder are trainable.
BN params (gamma, beta) and decoder params can safely share the same LR as they are
all learning to adapt to the new ELA domain.

In [ ]:
# ============================================================
# 5.1 Loss, Optimizer, Scheduler (All Trainable — From Scratch)
# ============================================================

bce_loss_fn = smp.losses.SoftBCEWithLogitsLoss()
dice_loss_fn = smp.losses.DiceLoss(mode='binary', from_logits=True)

def criterion(pred, target):
    return bce_loss_fn(pred, target) + dice_loss_fn(pred, target)

# All params trainable — no frozen encoder
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

print(f'Optimizer: AdamW (LR={LEARNING_RATE}, weight_decay=1e-5)')
print(f'  Total trainable: {sum(p.numel() for p in model.parameters()):,}')

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3)

In [ ]:
# ============================================================
# 5.3 Training Loop (with checkpoint save/resume + AMP)
# ============================================================

scaler = GradScaler('cuda')

history = {
    'train_loss': [], 'val_loss': [], 'val_pixel_f1': [], 'val_pixel_iou': [],
    'lr': []
}

best_val_loss = float('inf')
best_epoch = 0
patience_counter = 0
best_model_state = None
start_epoch = 1

if RESUME and os.path.exists(LATEST_CHECKPOINT):
    print(f'Checkpoint found: {LATEST_CHECKPOINT}')
    ckpt = torch.load(LATEST_CHECKPOINT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(DEVICE)
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt['best_val_loss']
    best_epoch = ckpt['best_epoch']
    patience_counter = ckpt['patience_counter']
    history = ckpt['history']
    if ckpt.get('best_model_state') is not None:
        best_model_state = ckpt['best_model_state']
    print(f'  Resuming from epoch {start_epoch} (best_epoch={best_epoch}, best_val_loss={best_val_loss:.4f})')
else:
    print('No checkpoint found \u2014 starting fresh.')

print(f'Starting training: epochs {start_epoch}-{EPOCHS}, patience={PATIENCE}')
print(f'LR: {LEARNING_RATE} | Input: Multi-Q ELA (Q={ELA_QUALITIES}) | AMP: Enabled')
print(f'{"="*80}')

for epoch in range(start_epoch, EPOCHS + 1):
    current_lr = optimizer.param_groups[0]['lr']
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, scaler)
    val_loss, val_f1, val_iou = validate(model, val_loader, criterion, DEVICE)
    scheduler.step(val_loss)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_pixel_f1'].append(val_f1)
    history['val_pixel_iou'].append(val_iou)
    history.get('lr', history.get('lr_encoder', [])).append(current_lr)
    if USE_WANDB:
        wandb.log({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                    'val_pixel_f1': val_f1, 'val_pixel_iou': val_iou, 'lr': current_lr})
    improved = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        torch.save(best_model_state, BEST_MODEL_PATH)
        improved = ' *'
    else:
        patience_counter += 1
    print(f'Epoch {epoch:>2}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
          f'Pixel F1: {val_f1:.4f} | IoU: {val_iou:.4f} | LR: {current_lr:.2e}{improved}')
    torch.save({
        'epoch': epoch, 'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_val_loss': best_val_loss, 'best_epoch': best_epoch,
        'patience_counter': patience_counter, 'best_model_state': best_model_state,
        'history': history,
    }, LATEST_CHECKPOINT)
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}. Best epoch: {best_epoch}')
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model = model.to(DEVICE)
    print(f'\nRestored best model from epoch {best_epoch} (val_loss={best_val_loss:.4f})')
else:
    print('\nNo improvement during training \u2014 using final weights')

print(f'{"="*80}')
print(f'Training complete. Best epoch: {best_epoch}, Best val loss: {best_val_loss:.4f}')

---

## 6. Evaluation

### Metrics Computed

**Pixel-Level (Localization):**
- Pixel F1 Score (= Dice coefficient)
- IoU (Intersection over Union / Jaccard index)
- Pixel AUC (ROC-AUC on per-pixel probabilities)

**Image-Level (Classification):**
- Accuracy (image classified as tampered if mask has any positive pixel above threshold)
- Per-class Precision, Recall, F1
- Macro F1
- ROC-AUC
- Confusion Matrix

In [ ]:
# ============================================================
# 6.1 Test Set Evaluation â€” Pixel-Level Metrics
# ============================================================

@torch.no_grad()
def evaluate_test(model, loader, device):
    """Full evaluation on test set."""
    model.eval()
    all_probs = []
    all_masks = []
    all_labels = []

    for images, masks, labels in tqdm(loader, desc='Test Eval'):
        images = images.to(device)
        predictions = model(images)
        probs = torch.sigmoid(predictions[0] if isinstance(predictions, tuple) else predictions)

        all_probs.append(probs.cpu().numpy())
        all_masks.append(masks.cpu().numpy())
        all_labels.extend(labels.numpy())

    all_probs = np.concatenate(all_probs, axis=0)  # (N, 1, H, W)
    all_masks = np.concatenate(all_masks, axis=0)   # (N, 1, H, W)
    all_labels = np.array(all_labels)

    return all_probs, all_masks, all_labels

test_probs, test_masks, test_labels = evaluate_test(model, test_loader, DEVICE)
test_preds_binary = (test_probs > 0.5).astype(np.float32)

# Pixel-level metrics (flatten all pixels)
pred_flat = test_preds_binary.flatten()
mask_flat = test_masks.flatten()
prob_flat = test_probs.flatten()

eps = 1e-7
tp = (pred_flat * mask_flat).sum()
fp = (pred_flat * (1 - mask_flat)).sum()
fn = ((1 - pred_flat) * mask_flat).sum()
tn = ((1 - pred_flat) * (1 - mask_flat)).sum()

pixel_f1 = (2 * tp) / (2 * tp + fp + fn + eps)
pixel_iou = tp / (tp + fp + fn + eps)
pixel_dice = pixel_f1  # Dice = F1 for binary
pixel_precision = tp / (tp + fp + eps)
pixel_recall = tp / (tp + fn + eps)

# Pixel AUC (subsample for speed if needed)
n_pixels = len(prob_flat)
if n_pixels > 5_000_000:
    sample_idx = np.random.choice(n_pixels, 5_000_000, replace=False)
    pixel_auc = roc_auc_score(mask_flat[sample_idx], prob_flat[sample_idx])
else:
    pixel_auc = roc_auc_score(mask_flat, prob_flat) if mask_flat.sum() > 0 and (1-mask_flat).sum() > 0 else 0.0

print(f'{"="*60}')
print(f'  PIXEL-LEVEL METRICS (Test Set)')
print(f'{"="*60}')
print(f'  Pixel Precision:  {pixel_precision:.4f}')
print(f'  Pixel Recall:     {pixel_recall:.4f}')
print(f'  Pixel F1 (Dice):  {pixel_f1:.4f}')
print(f'  Pixel IoU:        {pixel_iou:.4f}')
print(f'  Pixel AUC:        {pixel_auc:.4f}')
print(f'{"="*60}')

if USE_WANDB:
    wandb.log({'pixel_f1': pixel_f1, 'pixel_iou': pixel_iou,
               'pixel_precision': pixel_precision, 'pixel_recall': pixel_recall,
               'pixel_auc': pixel_auc})


In [ ]:
# ============================================================
# 6.2 Test Set Evaluation â€” Image-Level Classification
# ============================================================

# Derive image-level classification from masks:
# An image is classified as "tampered" if any predicted pixel > threshold
MASK_AREA_THRESHOLD = 100  # minimum number of tampered pixels to classify as tampered

image_pred_labels = []
image_pred_scores = []

for i in range(len(test_probs)):
    prob_map = test_probs[i, 0]  # (H, W)
    binary_map = (prob_map > 0.5).astype(np.float32)
    tampered_pixel_count = binary_map.sum()

    # Classification: tampered if enough pixels predicted as tampered
    pred_label = 1 if tampered_pixel_count >= MASK_AREA_THRESHOLD else 0
    image_pred_labels.append(pred_label)

    # Score: max probability in the mask (for ROC-AUC)
    image_pred_scores.append(prob_map.max())

image_pred_labels = np.array(image_pred_labels)
image_pred_scores = np.array(image_pred_scores)

# Classification metrics
cls_accuracy = accuracy_score(test_labels, image_pred_labels)
cls_report = classification_report(test_labels, image_pred_labels,
                                     target_names=['Authentic', 'Tampered'],
                                     output_dict=True)
cls_macro_f1 = f1_score(test_labels, image_pred_labels, average='macro')
cls_auc = roc_auc_score(test_labels, image_pred_scores) if len(np.unique(test_labels)) > 1 else 0.0

print(f'{"="*60}')
print(f'  IMAGE-LEVEL CLASSIFICATION (Test Set)')
print(f'{"="*60}')
print(f'  Test Accuracy:    {cls_accuracy:.4f}  ({cls_accuracy*100:.2f}%)')
print(f'  Macro F1:         {cls_macro_f1:.4f}')
print(f'  ROC-AUC:          {cls_auc:.4f}')
print(f'')
print(f'  Per-Class Results:')
print(f'  {"":>12} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Support":>10}')
for cls_name in ['Authentic', 'Tampered']:
    r = cls_report[cls_name]
    print(f'  {cls_name:>12} {r["precision"]:>10.4f} {r["recall"]:>10.4f} {r["f1-score"]:>10.4f} {r["support"]:>10.0f}')
print(f'{"="*60}')

# Classification report (full)
print('\nFull Classification Report:')
print(classification_report(test_labels, image_pred_labels, target_names=['Authentic', 'Tampered']))

if USE_WANDB:
    wandb.log({'image_accuracy': cls_accuracy, 'image_macro_f1': cls_macro_f1,
               'image_roc_auc': cls_auc})


In [ ]:
# ============================================================
# 6.3 Confusion Matrix and ROC Curve
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(test_labels, image_pred_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Authentic', 'Tampered'],
            yticklabels=['Authentic', 'Tampered'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix (Acc={cls_accuracy:.2%})')

# Print confusion details
tn, fp, fn, tp_cls = cm.ravel()
total = cm.sum()
print(f'Confusion Matrix:')
print(f'  TN={tn}, FP={fp}, FN={fn}, TP={tp_cls}')
print(f'  FP Rate: {fp/(tn+fp)*100:.1f}%')
print(f'  FN Rate: {fn/(fn+tp_cls)*100:.1f}%')

# ROC Curve
if len(np.unique(test_labels)) > 1:
    fpr, tpr, thresholds = roc_curve(test_labels, image_pred_scores)
    axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {cls_auc:.4f}')
    axes[1].plot([0, 1], [0, 1], 'r--', alpha=0.5)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'ROC Curve (AUC={cls_auc:.4f})')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.suptitle(f'{VERSION} â€” Classification Performance', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6.4 Training Curves
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss curves
axes[0, 0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss')
axes[0, 0].plot(epochs_range, history['val_loss'], 'r-', label='Val Loss')
axes[0, 0].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training vs Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Pixel F1
axes[0, 1].plot(epochs_range, history['val_pixel_f1'], 'g-', linewidth=2)
axes[0, 1].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Pixel F1')
axes[0, 1].set_title('Validation Pixel F1')
axes[0, 1].grid(True, alpha=0.3)

# Pixel IoU
axes[1, 0].plot(epochs_range, history['val_pixel_iou'], 'm-', linewidth=2)
axes[1, 0].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Pixel IoU')
axes[1, 0].set_title('Validation Pixel IoU')
axes[1, 0].grid(True, alpha=0.3)

# Learning Rate
axes[1, 1].plot(epochs_range, history.get('lr', history.get('lr_encoder', [])), 'k-', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Learning Rate')
axes[1, 1].set_title('Learning Rate Schedule')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f'{VERSION} â€” Training History', fontsize=14)
plt.tight_layout()
plt.show()

---

## 7. Visualization

### Original â†’ Ground Truth â†’ Predicted â†’ Overlay

The key deliverable: pixel-level tampered region predictions visualized alongside the original image and ground truth mask.

In [ ]:
# ============================================================
# 7.1 Original / Ground Truth / Predicted / Overlay Grid
# ============================================================

def visualize_predictions(model, dataset, indices, device, title='Predictions'):
    """Display Original | GT Mask | Predicted Mask | Overlay for each index."""
    n = len(indices)
    fig, axes = plt.subplots(n, 4, figsize=(20, 5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    model.eval()

    for row, idx in enumerate(indices):
        img_tensor, gt_mask, label = dataset[idx]

        # Predict
        with torch.no_grad():
            pred_logit = model(img_tensor.unsqueeze(0).to(device))
            pred_prob = torch.sigmoid(pred_logit).cpu().squeeze().numpy()

        pred_binary = (pred_prob > 0.5).astype(np.float32)

        # Load original RGB image for display
        rgb_path = dataset.image_paths[idx]
        rgb_img = Image.open(rgb_path).convert('RGB').resize((gt_mask.shape[-1], gt_mask.shape[-2]))
        rgb_display = np.array(rgb_img).astype(np.float32) / 255.0
        gt_display = gt_mask.squeeze(0).numpy()

        # Col 0: Original
        axes[row, 0].imshow(rgb_display)
        axes[row, 0].set_title(f'Original ({"Tampered" if label==1 else "Authentic"})', fontsize=11)
        axes[row, 0].axis('off')

        # Col 1: Ground Truth
        axes[row, 1].imshow(gt_display, cmap='hot', vmin=0, vmax=1)
        axes[row, 1].set_title(f'Ground Truth (sum={gt_display.sum():.0f})', fontsize=11)
        axes[row, 1].axis('off')

        # Col 2: Predicted Mask
        axes[row, 2].imshow(pred_binary, cmap='hot', vmin=0, vmax=1)
        axes[row, 2].set_title(f'Predicted (sum={pred_binary.sum():.0f})', fontsize=11)
        axes[row, 2].axis('off')

        # Col 3: Overlay
        overlay = rgb_display.copy()
        # Green for GT, Red for Predicted
        overlay_mask = np.zeros_like(overlay)
        overlay_mask[:, :, 1] = gt_display * 0.4       # Green = GT
        overlay_mask[:, :, 0] = pred_binary * 0.4       # Red = Predicted
        combined = np.clip(overlay * 0.6 + overlay_mask, 0, 1)
        axes[row, 3].imshow(combined)
        axes[row, 3].set_title('Overlay (green=GT, red=pred)', fontsize=11)
        axes[row, 3].axis('off')

    plt.suptitle(f'{VERSION} â€” {title}', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

# Select sample images: 3 tampered + 2 authentic from test set
tampered_indices = [i for i, l in enumerate(test_labels) if l == 1]
authentic_indices = [i for i, l in enumerate(test_labels) if l == 0]

sample_tp = tampered_indices[:14]
sample_au = authentic_indices[:6]

print('--- Tampered Image Predictions ---')
visualize_predictions(model, test_dataset, sample_tp, DEVICE, 'Tampered Images')

print('\n--- Authentic Image Predictions ---')
visualize_predictions(model, test_dataset, sample_au, DEVICE, 'Authentic Images')

if USE_WANDB:
    try:
        wandb.log({'prediction_examples': wandb.Image(plt.gcf())})
    except Exception:
        pass


In [ ]:
# ============================================================
# 7.2 Per-Image Metric Distribution
# ============================================================

# Compute per-image pixel F1
per_image_f1 = []
per_image_iou = []
per_image_labels = []

for i in range(len(test_probs)):
    pred = (test_probs[i].flatten() > 0.5).astype(np.float32)
    mask = test_masks[i].flatten()

    tp_i = (pred * mask).sum()
    fp_i = (pred * (1 - mask)).sum()
    fn_i = ((1 - pred) * mask).sum()

    f1_i = (2 * tp_i) / (2 * tp_i + fp_i + fn_i + 1e-7)
    iou_i = tp_i / (tp_i + fp_i + fn_i + 1e-7)

    per_image_f1.append(f1_i)
    per_image_iou.append(iou_i)
    per_image_labels.append(test_labels[i])

per_image_f1 = np.array(per_image_f1)
per_image_iou = np.array(per_image_iou)
per_image_labels = np.array(per_image_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 distribution by class
for cls, name in [(0, 'Authentic'), (1, 'Tampered')]:
    mask_cls = per_image_labels == cls
    axes[0].hist(per_image_f1[mask_cls], bins=30, alpha=0.6, label=f'{name} (n={mask_cls.sum()})')
axes[0].set_xlabel('Pixel F1 Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Per-Image Pixel F1 Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# IoU distribution by class
for cls, name in [(0, 'Authentic'), (1, 'Tampered')]:
    mask_cls = per_image_labels == cls
    axes[1].hist(per_image_iou[mask_cls], bins=30, alpha=0.6, label=f'{name} (n={mask_cls.sum()})')
axes[1].set_xlabel('Pixel IoU Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Per-Image Pixel IoU Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'{VERSION} â€” Per-Image Metric Distributions', fontsize=14)
plt.tight_layout()
plt.show()

# Summary stats
print(f'Per-Image Pixel F1:')
print(f'  Tampered:  mean={per_image_f1[per_image_labels==1].mean():.4f}, '
      f'std={per_image_f1[per_image_labels==1].std():.4f}')
print(f'  Authentic: mean={per_image_f1[per_image_labels==0].mean():.4f}, '
      f'std={per_image_f1[per_image_labels==0].std():.4f}')

---

## 8. Results Summary

In [ ]:
# ============================================================
# 8.1 Results Tracking Table
# ============================================================

print(f'{"="*100}')
print(f'  RESULTS SUMMARY — vR.P.41')
print(f'{"="*100}')
print()

print('Cross-Track Comparison:')
print(f'{"Version":<12} {"Track":<12} {"Encoder":<26} {"Input":<18} '
      f'{"Pixel-F1":<10} {"IoU":<8}')
print('-' * 94)

print(f'{"vR.P.3":<12} {"Pretrained":<12} {"ResNet-34":<26} {"ELA Q90":<18} '
      f'{"0.6920":<10} {"0.5291":<8}')
print(f'{"vR.P.10":<12} {"Pretrained":<12} {"ResNet-34":<26} {"ELA+CBAM":<18} '
      f'{"0.7277":<10} {"0.5719":<8}')
print(f'{"vR.P.19":<12} {"Pretrained":<12} {"ResNet-34":<26} {"MQ-RGB-ELA 9ch":<18} '
      f'{"0.7965":<10} {"0.6622":<8}')
print(f'{"vR.P.40.3":<12} {"Pretrained":<12} {"inception-v1-custom":<26} '
      f'{"MQ-RGB-ELA 9ch":<18} {"TBD":<10} {"TBD":<8}')
print(f'{"vR.P.40.4":<12} {"Pretrained":<12} {"inception-v2-custom":<26} '
      f'{"MQ-RGB-ELA 9ch":<18} {"TBD":<10} {"TBD":<8}')
print(f'{"vR.P.40.5":<12} {"Pretrained":<12} {"inception-v3-custom":<26} '
      f'{"MQ-RGB-ELA 9ch":<18} {"TBD":<10} {"TBD":<8}')

# This run
print(f'{"vR.P.41":<12} {"Pretrained":<12} {"unified-inception-v123":<26} '
      f'{"MQ-RGB-ELA 9ch":<18} '
      f'{pixel_f1:<10.4f} {pixel_iou:<8.4f}')
print()

print(f'Pixel-Level Metrics:')
print(f'  Pixel F1 (Dice): {pixel_f1:.4f}')
print(f'  Pixel IoU:       {pixel_iou:.4f}')
print(f'  Pixel Precision: {pixel_precision:.4f}')
print(f'  Pixel Recall:    {pixel_recall:.4f}')
print(f'  Pixel AUC:       {pixel_auc:.4f}')
print()
print(f'Image-Level Metrics:')
print(f'  Accuracy:        {cls_accuracy:.4f} ({cls_accuracy*100:.2f}%)')
print(f'  Macro F1:        {cls_macro_f1:.4f}')
print(f'  ROC-AUC:         {cls_auc:.4f}')

---

## 9. Discussion

### What Changed in vR.P.41

**Unified Inception V1+V2+V3 encoder trained from scratch**
- Progressive multi-generation architecture: V1 (early) → V2 (mid) → V3 (deep stages)
- Combines three Inception generations in one encoder for complementary feature extraction
- Ported from Hitansh Inception DeepFake notebook (TensorFlow/Keras) to PyTorch + SMP

### Architecture Design
- **V1 stages** (Stage 2): Classic 4-branch parallel (1x1, 3x3, 5x5, MaxPool), no BN
- **V2 stages** (Stage 3): Factorized 5x5 into two 3x3, BatchNorm, AvgPool branch
- **V3 stages** (Stages 4–5): Asymmetric 1xN + Nx1 factorization, BatchNorm
- Filter counts preserved exactly from Hitansh reference

### Key Differences from P.40.3/P.40.4/P.40.5
- Those use a single Inception type (V1 or V2 or V3) across all stages
- P.41 uses the *correct* historical progression: V1 → V2 → V3
- Tests whether combining all three types provides complementary features

### Expected Outcome
- Richer feature extraction from progressive multi-scale + multi-factorization
- Likely stronger than single-type encoders (P.40.3–40.5)
- Still from scratch (no pretrained weights), so compare fairly against P.40.x series

In [ ]:
# ============================================================
# 10. Save Model
# ============================================================

model_filename = f'{VERSION}_unet_unified_inception_model.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'best_epoch': best_epoch,
    'best_val_loss': best_val_loss,
    'history': history,
    'config': {
        'version': VERSION, 'encoder': ENCODER,
        'architecture': 'UNet-UnifiedInceptionV123',
        'image_size': IMAGE_SIZE, 'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE, 'ela_qualities': ELA_QUALITIES,
        'in_channels': IN_CHANNELS, 'epochs_trained': len(history['train_loss']),
        'seed': SEED,
    }
}, model_filename)

print(f'Model saved: {model_filename}')
print(f'File size: {os.path.getsize(model_filename) / 1e6:.1f} MB')
print(f'\n{VERSION} complete.')

# --- Save Experiment Artifacts ---
import json as _json
from datetime import datetime as _dt

_metrics = {
    'version': VERSION,
    'best_epoch': best_epoch,
    'best_val_loss': float(best_val_loss),
    'epochs_trained': len(history['train_loss']),
    'history': {k: [float(v) for v in vals] for k, vals in history.items()},
}
with open(os.path.join(RESULTS_DIR, f'{VERSION}_metrics.json'), 'w') as _f:
    _json.dump(_metrics, _f, indent=2)
print(f'Metrics saved to {RESULTS_DIR}/{VERSION}_metrics.json')

if USE_WANDB:
    try:
        wandb.log({
            'pixel_f1': pixel_f1, 'pixel_iou': pixel_iou,
            'pixel_precision': pixel_precision, 'pixel_recall': pixel_recall,
            'pixel_auc': pixel_auc, 'cls_accuracy': cls_accuracy,
            'cls_macro_f1': cls_macro_f1, 'cls_auc': cls_auc,
            'best_epoch': best_epoch, 'best_val_loss': best_val_loss,
        })
        wandb.finish()
        print('W&B run finished.')
    except Exception as e:
        print(f'W&B finish warning: {e}')